# KAT-VAD v2 — P2 / **D11: E2(b) + E2(c) re-read on DoTA-dev at stride 3** (no training)

E1 adopted protocol **B** (DoTA at stride 3, `core/docs/v2/RESULTS_E1.md`), so addendum **D2/D11** fires: the CRN
reference is chosen at the stride E1 adopts. This re-runs `core.tools.crn_select` — the same rule (proposal §4.2
steps 1–6, impl. choices I1–I9 in addendum §7) — with DoTA-dev read from `DoTA_s1_ncc[::3]`. **T2 is unchanged** (s8).

| step | what | CPU/GPU |
|---|---|---|
| 1 | preflight + stage the T2 CLIP cache and the DoTA-dev s1 features to VM disk | CPU |
| 2 | **regression gate**: the s8 path and the new s1 path at stride 8 must give the *same* read-out | CPU, ~minutes |
| 3 | the D11 read-out at stride 3 | CPU, ~minutes |
| 4 | record | — |

**Drive layout:** code + outputs in `Thesis-V2/`; data and caches in `Thesis/` (read only).
Read-outs go to `Thesis-V2/outputs/REPORTS/v2_E2_s3/`. A GPU is not needed.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert str(constants.OUTPUT_ROOT) == f'{DRIVE_V2}/outputs', constants.OUTPUT_ROOT

DATASET    = constants.DADA_ORIGIN_DATASET
T2         = constants.DATA_ROOT / DATASET
CENSUS     = T2 / 'counts' / 'resolved_census.json'
ANNOTATION = (constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME
              / constants.DADA_ORIGIN_ANNOTATION_FILENAME)
DADA_CLIP  = constants.CLIP_CACHE_DIR / DATASET
DOTA       = constants.DATA_ROOT / 'DoTA'
DOTA_S8    = constants.CLIP_CACHE_DIR / 'DoTA_s8_ncc'
DOTA_S1    = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
REPORT     = constants.OUTPUT_ROOT / 'REPORTS' / 'v2_E2_s3'
LOCAL      = Path('/content/e2s3')
REPORT.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)
for name, path in (('T2 meta', T2 / constants.META_FILENAME), ('census', CENSUS), ('annotation', ANNOTATION),
                   ('DADA clip', DADA_CLIP), ('DoTA s8', DOTA_S8), ('DoTA s1', DOTA_S1),
                   ('labels_s8', DOTA / 'labels_s8'), ('metadata', DOTA / 'metadata_val.json')):
    print(f'  {name:10s} {path}   {"OK" if path.exists() else "MISSING"}')

In [ ]:
%%bash
# crn_select imports core.data.* which pull av / einops / faiss transitively; openpyxl reads the xlsx.
pip install -q "transformers==4.56.*" av einops faiss-cpu openpyxl
python -c "import av, einops, faiss, openpyxl; print('deps OK')"

## 1. Preflight + stage to VM disk

In [ ]:
import json
import shutil
import subprocess
import time

addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert 'D11' in addendum and '## 10. P5 implementation choices' in addendum, (
    f'{REPO} predates the D11 / P5 code -- re-upload the v2 code')
from core.data.v2_splits import load_split  # noqa: E402
from core.tools import crn_select  # noqa: E402

assert hasattr(crn_select, 'load_dota_dev_s1'), 'this upload has no --dota-s1-dir path'
dev = load_split(constants.V2_SPLIT_DOTA_DEV)
for path in (T2 / constants.META_FILENAME, CENSUS, ANNOTATION, DADA_CLIP, DOTA_S8, DOTA_S1,
             DOTA / 'metadata_val.json', DOTA / 'val_split.txt'):
    assert path.exists(), f'missing {path}'

t0 = time.time()
L_DADA = LOCAL / 'dada_clip'
if not L_DADA.exists():
    shutil.copytree(DADA_CLIP, L_DADA)
L_S1, L_S8 = LOCAL / 'dota_s1_dev', LOCAL / 'dota_s8_dev'
for src, dst in ((DOTA_S1, L_S1), (DOTA_S8, L_S8)):
    dst.mkdir(exist_ok=True)
    for v in dev:
        if not (dst / f'{v}.npy').exists():
            shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
print(f'staged in {time.time() - t0:.0f}s: DADA {len(list(L_DADA.glob("*.npy")))} files, '
      f'DoTA-dev s1 {len(list(L_S1.glob("*.npy")))}, s8 {len(list(L_S8.glob("*.npy")))}')

T2_ARGS = ['--t2-meta', T2 / constants.META_FILENAME, '--annotation', ANNOTATION,
           '--census', CENSUS, '--dada-clip-dir', L_DADA]
S1_ARGS = ['--dota-s1-dir', L_S1, '--dota-metadata', DOTA / 'metadata_val.json',
           '--dota-split-file', DOTA / 'val_split.txt']


def run_select(extra, out_dir):
    cmd = [sys.executable, '-m', 'core.tools.crn_select', *T2_ARGS, *extra, '--out-dir', out_dir]
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print(''.join(tail))
        raise RuntimeError(f'crn_select exited {proc.returncode}')
    return json.loads((Path(out_dir) / crn_select.READOUT_JSON).read_text())

## 2. Regression gate — the new s1 path must reproduce the s8 path at stride 8

Same features (`s1[::8] == DoTA_s8_ncc`, verified 30/30 on 2026-09-29) and the same labels (annotation rounded onto
the same length) ⇒ **identical** references, choice, transfer and verdict. Any difference stops the notebook:
the s1 path would be measuring something other than a stride change.

In [ ]:
import math

ref_s8 = run_select(['--dota-labels-dir', DOTA / 'labels_s8', '--dota-clip-dir', L_S8], REPORT / 'gate_s8_path')
ref_s1 = run_select([*S1_ARGS, '--dota-stride', '8'], REPORT / 'gate_s1_stride8')


def diffs(a, b, path=''):
    out = []
    if isinstance(a, dict) and isinstance(b, dict):
        for k in sorted(set(a) | set(b)):
            if k not in a or k not in b:
                out.append(f'{path}/{k}: only in {"s8" if k in a else "s1"}')
            else:
                out += diffs(a[k], b[k], f'{path}/{k}')
    elif isinstance(a, list) and isinstance(b, list) and len(a) == len(b):
        for i, (x, y) in enumerate(zip(a, b)):
            out += diffs(x, y, f'{path}[{i}]')
    elif isinstance(a, float) and isinstance(b, float):
        if not (math.isclose(a, b, rel_tol=0, abs_tol=1e-9) or (math.isnan(a) and math.isnan(b))):
            out.append(f'{path}: s8 {a!r} vs s1 {b!r}')
    elif a != b:
        out.append(f'{path}: s8 {a!r} vs s1 {b!r}')
    return out


# The only intended differences: the DoTA corpus name ('DoTA-dev' vs 'DoTA-dev-s8') and 'dota_stride'.
norm = json.loads(json.dumps(ref_s1).replace('DoTA-dev-s8', 'DoTA-dev'))
compared = ('references', 'choice', 'transfer', 'verdict', 'position', 'e2a_share', 'coverage')
problems = [d for key in compared for d in diffs(ref_s8.get(key), norm.get(key), key)]
print(f'{len(problems)} differences')
for p in problems[:30]:
    print('  ', p)
assert not problems, 'REGRESSION GATE FAILED -- do not read step 3; send this output to Claude'
print('GATE PASSED: s1 path at stride 8 == s8 path. Verdict at s8:', ref_s8['verdict'])

## 3. D11 read-out — DoTA-dev at stride 3

The choice rule is applied mechanically (proposal §4.2 steps 4–5 on DoTA-dev `r`, E2(c) veto). Per addendum D2 the
**stride-3 verdict is the CRN reference P5 uses**. Remember E2(b)'s caveat (`RESULTS_E2_CRN.md`): print `−f` beside `r`.

In [ ]:
out3 = run_select([*S1_ARGS, '--dota-stride', '3'], REPORT / 's1_stride3')
print((REPORT / 's1_stride3' / crn_select.READOUT_MD).read_text())
print('\nVERDICT at stride 3:', out3['verdict'], '| at stride 8:', ref_s8['verdict'])

## 4. Record

Paste `Thesis-V2/outputs/REPORTS/v2_E2_s3/s1_stride3/e2_readout.md` (and the verdict line above) back to Claude.